# Тема 1. Анализ данных с Pandas

## Занятие 1 (теория)

**План занятия:**
1. Series и DataFrame — базовые структуры Pandas
2. Знакомство с датасетом: загрузка, первый осмотр
3. `value_counts()`
4. Сортировка: `sort_values()`
5. Индексация и извлечение данных: `loc`/`iloc`, булева индексация
6. Применение функций: `apply()`, `map()`
7. Группировка: `groupby()` + `agg()`
8. Сводные таблицы: `crosstab()`, `pivot_table()`
9. Добавление и удаление колонок
10. Пропуски в данных: `isna`, `fillna`, `dropna`
11. Итоговый мини-кейс: от разведочного анализа к первому правилу-предсказанию

**Практика** по этой теме — в отдельном ноутбуке `lesson01_pandas_practice.ipynb` (датасет Titanic).

**Дополнительные темы** (NumPy, объединение таблиц, работа с датами) — в `lesson01_pandas_advanced.ipynb`.

---
## 1. Series и DataFrame

В Pandas есть две основные структуры данных:

- **`Series`** — одномерный массив значений с индексом (подписями к каждому элементу). Можно думать про него как про один столбец таблицы или как про `dict`, но с гарантированным порядком и удобными операциями.
- **`DataFrame`** — двумерная таблица: набор `Series`, у которых общий индекс (строки), и каждая колонка — это `Series` своего типа. Строки — это объекты (наблюдения), столбцы — это признаки.

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 2)

1. Создайте `Series` из списка `[400, 300, 200, 250]` с именованным индексом `["Andrew", "Bob", "Charles", "Ann"]`, сохраните в `salaries`, выведите.
2. Отфильтруйте `salaries`, оставив только значения больше 250 (сравнение `salaries > 250` даёт булеву маску, ей же можно сразу индексировать).

In [17]:
salaries = pd.Series([500,300,200,250], index = ["Andrew", "Bob", "Charles", "Ann"])

slaries[salaries > 250]

Andrew    500
Bob       300
dtype: int64

Cоздайте `DataFrame` из словаря — по ключу на колонку: `name` (список имён), `salary` (числа), `is_manager` (True/False). Сохраните в `df_toy`, выведите, обратите внимание на разные типы колонок.

In [51]:
df_toy = pd.DataFrame({0: [1,2,3,'dt'], 1: range(4)})
df_toy.info()

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       4 non-null      object
 1   1       4 non-null      int64 
dtypes: int64(1), object(1)
memory usage: 196.0+ bytes


Теперь перейдём к работе с настоящими данными.

---
## 2. Знакомство с датасетом

Будем работать с датасетом телеком-оператора: по каждому клиенту известны его тарифные опции и статистика звонков, а также признак `Churn` — ушёл ли клиент от оператора (*отток*, churn). Это реальная задача бизнеса: предсказать отток заранее и попытаться удержать клиента.

Загрузим данные (`read_csv`) и посмотрим на первые строки (`head`):

In [36]:
df = pd.read_csv("../data/telecom_churn.csv")
df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,KS,128,415,No,Yes,25,265.1,110,45.07,197.4,99,16.78,244.7,91,11.01,10.0,3,2.70,1,False
1,OH,107,415,No,Yes,26,161.6,123,27.47,195.5,103,16.62,254.4,103,11.45,13.7,3,3.70,1,False
2,NJ,137,415,No,No,0,243.4,114,41.38,121.2,110,10.30,162.6,104,7.32,12.2,5,3.29,0,False
3,OH,84,408,Yes,No,0,299.4,71,50.90,61.9,88,5.26,196.9,89,8.86,6.6,7,1.78,2,False
4,OK,75,415,Yes,No,0,166.7,113,28.34,148.3,122,12.61,186.9,121,8.41,10.1,3,2.73,3,False


**Задание:** выведите размерность датасета и список колонок.

Подсказка: `.shape` — кортеж (число строк, число столбцов), `.columns` — названия колонок.

In [40]:
df.columns, df.index


(Index(['State', 'Account length', 'Area code', 'International plan',
        'Voice mail plan', 'Number vmail messages', 'Total day minutes',
        'Total day calls', 'Total day charge', 'Total eve minutes',
        'Total eve calls', 'Total eve charge', 'Total night minutes',
        'Total night calls', 'Total night charge', 'Total intl minutes',
        'Total intl calls', 'Total intl charge', 'Customer service calls',
        'Churn'],
       dtype='str'),
 RangeIndex(start=0, stop=3333, step=1))

Вызовите `.info()` — показывает типы всех колонок и есть ли пропуски (сравнивает число непустых значений с числом строк).

In [53]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   3333 non-null   str    
 1   Account length          3333 non-null   int64  
 2   Area code               3333 non-null   int64  
 3   International plan      3333 non-null   str    
 4   Voice mail plan         3333 non-null   str    
 5   Number vmail messages   3333 non-null   int64  
 6   Total day minutes       3333 non-null   float64
 7   Total day calls         3333 non-null   int64  
 8   Total day charge        3333 non-null   float64
 9   Total eve minutes       3333 non-null   float64
 10  Total eve calls         3333 non-null   int64  
 11  Total eve charge        3333 non-null   float64
 12  Total night minutes     3333 non-null   float64
 13  Total night calls       3333 non-null   int64  
 14  Total night charge      3333 non-null   float64
 15

Признак `Churn` имеет тип `bool`. Для дальнейших вычислений (сравнений по числовым порогам, красивых таблиц с 0/1) удобнее привести его к `int64`:

In [52]:
df["Churn"] = df["Churn"].astype("int64")

`.describe()` — статистика по числовым колонкам (count, mean, std, min, квартили, max). Чтобы посмотреть на категориальные и булевы колонки — нужно явно указать `include` (в pandas 3.x строковые колонки имеют тип `str`; в версиях до 3.0 — `object`, тогда используйте его вместо `str`):

In [63]:
df.describe(include=['str'])

,State,International plan,Voice mail plan
count,3333,3333,3333
unique,51,2,2
top,WV,No,No
freq,106,3010,2411


---
## 3. `value_counts()`

Для категориальных и булевых признаков полезно посмотреть на распределение значений — сколько раз встречается каждое, и с параметром `normalize=True` — какую долю составляет.

**Задание преподавателю на занятии:**
1. `df['Churn'].value_counts()` — абсолютные числа.
2. `df['Churn'].value_counts(normalize=True)` — доли.

In [69]:
# df['Churn'].value_counts()
# df['Churn'].value_counts(normalize=True)
df['Churn'].value_counts(normalize=True)

Churn
0    0.86
1    0.14
Name: proportion, dtype: float64

Так как `Churn` уже 0/1, тот же процент можно получить и одной строкой — `.mean()` по бинарному признаку сразу даёт долю единиц:

In [74]:
round(float(df["Churn"].mean() * 100), 1)


14.5

14.5% клиентов ушли — довольно высокий отток для телеком-бизнеса. Запомним эту цифру, она пригодится в финальном кейсе.

---
## 4. Сортировка: `sort_values()`

`DataFrame` можно отсортировать по значению одной или нескольких колонок (`ascending=False` — по убыванию).

**Задание:**
1. Отсортируйте клиентов по убыванию `Total day charge` (плата за дневные звонки), выведите первые 5.
2. Отсортируйте сразу по двум колонкам: сначала по `Churn`, потом по `Total day charge` (по убыванию), с разным направлением сортировки для каждой (`ascending=[True, False]`).

In [78]:
# df.sort_values(by=..., ascending=...).head()
df.sort_values(by=['Churn', 'Total day charge'])

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
1397,VT,101,510,No,No,0,0.0,0,0.00,192.1,119,16.33,168.8,95,7.60,7.2,4,1.94,1,0
2736,OK,127,510,No,Yes,27,2.6,113,0.44,254.0,102,21.59,242.7,156,10.92,9.2,5,2.48,3,0
2753,OH,134,415,No,No,0,7.8,86,1.33,171.4,100,14.57,186.5,80,8.39,12.9,2,3.48,2,0
1986,WI,70,415,No,No,0,7.9,100,1.34,136.4,83,11.59,156.6,89,7.05,12.1,1,3.27,0,0
1052,OR,98,415,Yes,No,0,12.5,67,2.13,256.6,90,21.81,169.4,88,7.62,7.7,9,2.08,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
605,MO,112,415,No,No,0,335.5,77,57.04,212.5,109,18.06,265.0,132,11.93,12.7,8,3.43,2,1
156,OH,83,415,No,No,0,337.4,120,57.36,227.4,116,19.33,153.9,114,6.93,15.8,7,4.27,0,1
2594,OH,115,510,Yes,No,0,345.3,81,58.70,203.4,106,17.29,217.5,107,9.79,11.8,8,3.19,1,1
985,NY,64,415,Yes,No,0,346.8,55,58.96,249.5,79,21.21,275.4,102,12.39,13.3,9,3.59,1,1


---
## 5. Индексация и извлечение данных

Разные способы получить нужный кусок таблицы:

- **Одна колонка**: `df['col']` (или `df.col`) → возвращает `Series`
- **Несколько колонок**: `df[['col1', 'col2']]` → возвращает `DataFrame`
- **`loc`** — по меткам (названиям индекса/колонок), **включает** правую границу среза
- **`iloc`** — по позициям (числам, как в обычном срезе Python), **не включает** правую границу

**Задание:** выведите одну колонку `'Total day minutes'` разными способами.

In [ ]:
# df['Total day minutes']


**Булева индексация** — способ отфильтровать строки по условию: `df[P(df['col'])]`, где `P` — логическое условие.

**Задание:** сравните, сколько в среднем говорят днём (`Total day minutes`) клиенты, которые ушли (`Churn == 1`), и те, кто остался (`Churn == 0`).

Ожидаемо: у ушедших заметно больше — примерно 207 минут против 175.

In [ ]:
# df[df["Churn"] == 1]["Total day minutes"].mean()
# df[df["Churn"] == 0]["Total day minutes"].mean()


**Важно:** при нескольких условиях используем `&` (И), `|` (ИЛИ), `~` (НЕ) — **не** `and`/`or`/`not`, и **каждое условие обязательно в скобках**:

```python
# Правильно:
df[(df['Churn'] == 0) & (df['International plan'] == 'No')]

# Неправильно — упадёт с ошибкой приоритета операторов:
df[df['Churn'] == 0 & df['International plan'] == 'No']
```

Найдём максимальную длительность международных звонков среди лояльных клиентов (`Churn == 0`) без подключённого международного тарифа (`International plan == 'No'`). Допишите пропуск — второе условие:

In [ ]:
df[(df["Churn"] == 0) & (____)]["Total intl minutes"].max()

`loc` — по меткам, включая правую границу. `iloc` — по позициям, как обычный питоновский срез.

**Задание:** сранить выводы по "State":"Area code" с помощью loc и iloc.

In [ ]:
# df.loc[0:5, "State":"Area code"]
# df.iloc[0:5, 0:3]


---
## 6. Применение функций: `apply()` и `map()`

`apply()` применяет функцию к каждой колонке (по умолчанию) или к каждой строке (`axis=1`). Часто используется с `lambda`.

**Задание:** отберите клиентов из штатов, название которых начинается на `'W'` — `df[df['State'].apply(lambda state: state[0] == 'W')]`.

In [ ]:
# df[df["State"].apply(lambda state: state[0] == "W")].head()


`map()` заменяет значения колонки по словарю `{старое: новое}`. Переведём `International plan` из текста `Yes`/`No` в булев тип — пригодится в финальном кейсе:

In [5]:
d = {"No": False, "Yes": True}
df["International plan"] = df["International plan"].map(d)
df["International plan"].head()

0    False
1    False
2    False
3     True
4     True
Name: International plan, dtype: bool

> **Чем `map` отличается от `replace`?** `replace` не трогает значения, которых нет в словаре, а `map` заменяет их на `NaN`. Если словарь покрывает не все значения колонки — используйте `replace`, а не `map`.

---
## 7. Группировка: `groupby()` + `agg()`

Общая схема (*split-apply-combine*):

```python
df.groupby(by=grouping_columns)[columns_to_show].function()
```

1. **Split** — делим строки на группы по значению `grouping_columns`
2. **Apply** — считаем что-то по каждой группе
3. **Combine** — собираем результат в одну таблицу

Сравним статистику по разговорам у ушедших и лояльных клиентов сразу несколькими агрегатами через `.agg()` — синтаксис с несколькими функциями сразу легко забыть, поэтому целиком:

In [6]:
columns_to_show = ["Total day minutes", "Total eve minutes", "Total night minutes"]

df.groupby(["Churn"])[columns_to_show].agg(["mean", "std", "min", "max"])

Total day minutes                    Total eve minutes               \
                   mean    std  min    max              mean    std   min   
Churn                                                                       
0                175.18  50.18  0.0  315.6            199.04  50.29   0.0   
1                206.91  69.00  0.0  350.8            212.41  51.73  70.9   

             Total night minutes                      
         max                mean    std   min    max  
Churn                                                 
0      361.8              200.13  51.11  23.2  395.0  
1      363.7              205.23  47.13  47.4  354.9

Видно, что ушедшие клиенты в среднем говорят днём заметно дольше (206 минут против 175) — возможно, это связано с более высокими счетами.

---
## 8. Сводные таблицы: `crosstab()` и `pivot_table()`

**`crosstab`** — таблица сопряжённости: сколько наблюдений приходится на каждую комбинацию значений двух категориальных признаков. Разберём на первом примере целиком, дальше будет только `pivot_table` — по аналогии:

In [7]:
pd.crosstab(df["Churn"], df["International plan"], margins=True)

International plan,False,True,All
Churn,,,
0,2664,186,2850
1,346,137,483
All,3010,323,3333


Видно, что среди клиентов с международным тарифом отток заметно выше (137 из 323 ≈ 42%), чем без него (186 из 3010 ≈ 6%).

**`pivot_table`** — обобщение `crosstab`: можно агрегировать любую функцию по числовой колонке (`values`), группируя по одной или нескольким (`index`/`columns`). Посчитаем средние `Total day/eve/night calls` по каждому `Area code`. Допишите функцию агрегации (по умолчанию она и так `"mean"`, но укажем явно):

In [ ]:
df.pivot_table(
    values=["Total day calls", "Total eve calls", "Total night calls"],
    index=["Area code"],
    aggfunc=____,
)

---
## 9. Добавление и удаление колонок

Новую колонку можно посчитать из существующих и сразу присвоить.

**Задание преподавателю на занятии:**
1. Создайте колонку `Total calls` — сумма `Total day calls`, `Total eve calls`, `Total night calls`, `Total intl calls`.
2. Выведите `df.head()`, чтобы убедиться, что колонка появилась.
3. Удалите её обратно — `df.drop(columns=[...], inplace=True)` (или `axis=1`).

In [ ]:
# df['Total calls'] = ...


In [ ]:
# df.drop(...)


---
## 10. Пропуски в данных

В нашем датасете пропусков нет (мы это видели в `.info()`), но на практике они встречаются почти всегда. Создадим копию датасета и искусственно внесём пропуски, чтобы отработать приёмы:

- `.isna()` / `.isnull()` — где пропуск (True/False)
- `.fillna(value)` — заполнить пропуски
- `.dropna()` — выбросить строки/столбцы с пропусками

Код внесения искусственных пропусков непоказателен сам по себе — приведём целиком, чтобы не терять время на занятии:

In [8]:
rng = np.random.default_rng(42)
df_missing = df.copy()

idx_num = rng.choice(df_missing.index, size=150, replace=False)
df_missing.loc[idx_num, "Total day minutes"] = np.nan

idx_cat = rng.choice(df_missing.index, size=80, replace=False)
df_missing.loc[idx_cat, "Voice mail plan"] = np.nan

df_missing.isna().sum()[df_missing.isna().sum() > 0]

Voice mail plan       80
Total day minutes    150
dtype: int64

In [9]:
median_minutes = df_missing["Total day minutes"].median()
df_missing["Total day minutes"] = df_missing["Total day minutes"].fillna(median_minutes)

df_missing = df_missing.dropna(subset=["Voice mail plan"])

df_missing.isna().sum().sum()

np.int64(0)

Числовой признак заполнили медианой (устойчива к выбросам), а строки с пропуском в категориальном признаке — просто выбросили, раз восстановить категорию нечем.

---
## 11. Итоговый мини-кейс: от EDA к первому правилу

Давайте попробуем предсказать отток «руками», без машинного обучения — этот же приём мы позже будем повторять с деревьями решений.

**Задание преподавателю на занятии:** постройте `pd.crosstab(df['Churn'], df['Customer service calls'], margins=True)` — как отток связан с числом обращений в поддержку? (Похоже, что начиная примерно с 4 обращений отток резко растёт.)

In [ ]:
# pd.crosstab(df["Churn"], df["Customer service calls"], margins=True)


Добавим бинарный признак «много обращений» и посмотрим на связь с оттоком одновременно с международным тарифом — здесь код важен для дальнейшего, приводим целиком:

In [10]:
df["Many_service_calls"] = (df["Customer service calls"] > 3).astype("int64")
df["Risky_combo"] = df["Many_service_calls"].astype(bool) & df["International plan"]

pd.crosstab(df["Risky_combo"], df["Churn"])

Churn,0,1
Risky_combo,,
False,2841,464
True,9,19


Сформулируем простое правило:

> Если у клиента подключён международный тариф **и** он обращался в поддержку больше 3 раз — предсказываем `Churn = 1`, иначе `Churn = 0`.

Посчитаем точность такого правила:

In [11]:
rule_predicts_churn = (df["Many_service_calls"] == 1) & df["International plan"]
accuracy = (rule_predicts_churn.astype(int) == df["Churn"]).mean()
round(float(accuracy * 100), 1)

85.8

Получилось **85.8%** — чуть лучше, чем наивное «все клиенты лояльны» (85.5%, это доля `Churn == 0`). Немного, но уже что-то, и мы получили это простым разведочным анализом — без единой модели МО.

Это будет нашей отправной точкой (*baseline*): любая обученная модель, которая не превосходит такое простое правило, скорее всего не стоит своей сложности. На следующих занятиях мы разберём, как алгоритм дерева решений находит подобные правила автоматически, по всем признакам сразу, а не только по двум выбранным вручную.

---
## Итог занятия

Сегодня разобрали:

- `Series`, `DataFrame` — базовые структуры
- `read_csv`, `head`, `info`, `describe`, `astype`
- `value_counts`
- `sort_values`
- `loc`/`iloc`, булева индексация (`&`, `|`, `~`, обязательные скобки)
- `apply`, `map`
- `groupby` + `agg`
- `crosstab`, `pivot_table`
- добавление/удаление колонок
- `isna`, `fillna`, `dropna`

**Дальше:**
- Практика по этой теме — `lesson01_pandas_practice.ipynb` (Titanic)
- Доп. материалы (NumPy, объединение таблиц, работа с датами) — `lesson01_pandas_advanced.ipynb`
- Следующее занятие — визуализация данных (matplotlib/seaborn)